In [1]:
import numpy as np
import pandas as pd


In [2]:
def download():
    
    from urllib.request import urlretrieve
    import os

    url = (
        "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/"
        "Maji_Ndogo/Maji_Ndogo_farm_survey_small.db"
    )

    db_file = "Maji_Ndogo_farm_survey_small.db"

    # Download only if the database is not already present
    if not os.path.exists(db_file):
        urlretrieve(url, db_file)
        print(f"Downloaded '{db_file}'.")
    else:
        print(f"'{db_file}' already exists.")

    return

In [3]:
def db_base():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sqlalchemy import create_engine, text

    engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')
    sql_query = """
    SELECT *
    FROM geographic_features
    LEFT JOIN weather_features USING (Field_ID)
    LEFT JOIN soil_and_crop_features USING (Field_ID)
    LEFT JOIN farm_management_features USING (Field_ID)
    """
    with engine.connect() as connection:
        MD_agric_df = pd.read_sql_query(text(sql_query), connection)

    # In this survey, the Crop_type and Annual_yield columns are stored in each
    # other's place. Swap them back so Crop_type holds crop names and Annual_yield
    # holds the numeric tonnage.
    MD_agric_df.rename(columns={'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, inplace=True)
    MD_agric_df.rename(columns={'Crop_type_Temp': 'Crop_type'}, inplace=True)

    # A handful of elevations were logged as negative — fix the sign.
    MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()

    # Clean known crop-name typos (e.g. 'cassaval' -> 'cassava').
    corrections = {'cassaval': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
    MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].apply(
        lambda crop: corrections.get(str(crop).strip(), str(crop).strip())
    )
    print(f'Loaded: {MD_agric_df.shape[0]} rows x {MD_agric_df.shape[1]} columns')
    return (MD_agric_df,)


In [4]:
MD_agric_df = db_base()[0]
MD_agric_df.head()

Loaded: 5654 rows x 18 columns


,Field_ID,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield
0,40734,786.05580,-7.389911,-7.556202,Rural_Akatsi,14.795113,1125.2,-3.1,33.1,15.00,0.62,Sandy,6.169393,0.085267,1.3,0.751354,cassava,0.577964
1,30629,674.33410,-7.736849,-1.051539,Rural_Sokoto,11.374611,1450.7,-3.9,30.6,13.35,0.64,Volcanic,5.676648,0.399684,2.2,1.069865,cassava,0.486302
2,39924,826.53390,-9.926616,0.115156,Rural_Sokoto,11.339692,2208.9,-1.8,28.4,13.30,0.69,Volcanic,5.331993,0.358029,3.4,2.208801,tea,0.649647
3,5754,574.94617,-2.420131,-6.592215,Rural_Kilimani,7.109855,328.8,-5.8,32.2,13.20,0.54,Loamy,5.328150,0.286687,2.4,1.277635,cassava,0.532348
4,14146,886.35300,-3.055434,-7.952609,Rural_Kilimani,55.007656,785.2,-2.5,31.0,14.25,0.72,Sandy,5.721234,0.043190,1.5,0.832614,wheat,0.555076


## __Challenge 1: Dataset Summary Metrics__

Every analysis starts by understanding what you are working with. Before building complex charts, let's extract the essential dimensions: how many columns the joined survey has, and the average elevation across all fields.

### __Task__
Write a function `get_dataset_summary` that accepts the dataset and returns a tuple `(number_of_columns, mean_elevation)`, where `number_of_columns` is an `int` and `mean_elevation` is the mean of the `Elevation` column.

> ⚠️ Do not change the function name `get_dataset_summary`.

### __Expected Output__
    ```
(18, 637.7907086314113)

In [5]:
def get_dataset_summary(df):
    """
    Returns a tuple containing:
    (number_of_columns, mean_elevation)

    Parameters
    ----------
    df : pandas.DataFrame
        The joined survey dataset.

    Returns
    -------
    tuple
        (number_of_columns, mean_elevation)
    """
    number_of_columns = df.shape[1]
    mean_elevation = df["Elevation"].mean()

    return (number_of_columns, mean_elevation)

In [6]:
summary = get_dataset_summary(MD_agric_df)
print(summary)

(18, np.float64(637.7907086314113))


## __Challenge 2: Rainfall by Location__

The overall `Rainfall` distribution shows multiple peaks — a sign that different sub-populations are being averaged together. Breaking it down by province reveals the hidden structure, and tells the audit which regions are genuinely dry and which are wet.

### __Task__
Write a function `get_mean_rainfall_by_location` that calculates the mean `Rainfall` grouped by `Location` and returns it as a pandas Series sorted in **ascending** order of values.

> ⚠️ Do not change the function name `get_mean_rainfall_by_location`.

### __Expected Output__
    ```
    Location
    Rural_Amanzi       723.628958
    Rural_Kilimani     791.352426
    Rural_Hawassa     1325.941003
    Rural_Akatsi      1584.884457
    Rural_Sokoto      1705.079431
    Name: Rainfall, dtype: float64
    
    ```

In [7]:
def get_mean_rainfall_by_location(df):
    """
    Calculate the mean rainfall for each location.

    Parameters
    ----------
    df : pandas.DataFrame
        The joined survey dataset.

    Returns
    -------
    pandas.Series
        Mean rainfall grouped by location, sorted in ascending order.
    """
    return (
        df.groupby("Location")["Rainfall"]
          .mean()
          .sort_values()
    )

In [8]:
mean_rainfall = get_mean_rainfall_by_location(MD_agric_df)
print(mean_rainfall)

Location
Rural_Amanzi       723.628958
Rural_Kilimani     791.352426
Rural_Hawassa     1325.941003
Rural_Akatsi      1584.884457
Rural_Sokoto      1705.079431
Name: Rainfall, dtype: float64


## __Challenge 3: Crop Distribution Volume__

Before diving into continuous relationships, it helps to understand the categorical landscape. How many fields are planted with each crop type? This Series drives both a bar chart and a pie chart.

### __Task__
Write a function `get_crop_counts` that counts the number of fields per `Crop_type` and returns a pandas Series sorted in **descending** order of counts.

> ⚠️ Do not change the function name `get_crop_counts`.

### __Expected Output__
    ```
    Crop_type
    wheat      1342
    tea         913
    potato      823
    cassava     672
    banana      633
    coffee      607
    maize       399
    rice        265
    Name: count, dtype: int64

    ```

In [9]:
def get_crop_counts(df):
    """
    Count the number of fields for each crop type.

    Parameters
    ----------
    df : pandas.DataFrame
        The joined survey dataset.

    Returns
    -------
    pandas.Series
        Crop counts sorted in descending order.
    """
    return (
        df["Crop_type"]
        .value_counts()
        .sort_values(ascending=False)
    )

In [10]:
crop_counts = get_crop_counts(MD_agric_df)
print(crop_counts)

Crop_type
wheat      1342
tea         913
potato      823
cassava     672
banana      633
coffee      607
maize       399
rice        265
Name: count, dtype: int64


- Rice, maize, and coffee are the three least common crops
- The three least common crops together total 1,271 fields
- Wheat's count alone (1,342) exceeds the combined total of the three least common crops

## __Challenge 4: Crop Median Yields__

A violin plot shows the full distribution of a continuous variable across categories. Let's isolate the precise median annual yields across crop varieties to locate the performance peaks.

### __Task__
Write a function `get_median_yield_by_crop` that calculates the median `Annual_yield` grouped by `Crop_type` and returns a pandas Series sorted in **descending** order of median yield.

> ⚠️ Do not change the function name `get_median_yield_by_crop`.

### __Expected Output__
    ```
    Crop_type
    rice       1.734914
    tea        1.657955
    cassava    1.447799
    maize      1.446741
    coffee     1.405154
    banana     1.391763
    wheat      1.389082
    potato     1.283974
    Name: Annual_yield, dtype: float64

    ```

In [11]:
def get_median_yield_by_crop(df):
    """
    Calculate median annual yield for each crop type.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing Crop_type and Annual_yield columns.

    Returns
    -------
    pandas.Series
        Median Annual_yield per Crop_type sorted descending.
    """
    
    median_yields = (
        df.groupby("Crop_type")["Annual_yield"]
        .median()
        .sort_values(ascending=False)
    )
    
    return median_yields

In [12]:
median_yields = get_median_yield_by_crop(MD_agric_df)

print(median_yields)

Crop_type
rice       1.734914
tea        1.657955
cassava    1.447799
maize      1.446741
coffee     1.405154
banana     1.391763
wheat      1.389082
potato     1.283974
Name: Annual_yield, dtype: float64


##  __Challenge 5: Temperature and Elevation Correlation__

Scatter plots reveal how two continuous variables relate. Let's quantify the linear connection between altitude and average ambient temperature with a single Pearson coefficient.

### __Task__
    Write a function `get_elevation_temp_correlation` that computes the Pearson correlation coefficient between `Elevation` and `Ave_temps`, returned as a `float`.

> ⚠️ Do not change the function name `get_elevation_temp_correlation`.

### __Expected Output__
    ```
    0.20306114718387722

    ```

In [13]:
def get_elevation_temp_correlation(df):
    """
    Calculate Pearson correlation between Elevation and Ave_temps.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing Elevation and Ave_temps columns.

    Returns
    -------
    float
        Pearson correlation coefficient.
    """
    
    correlation = df["Elevation"].corr(df["Ave_temps"])
    
    return float(correlation)

In [14]:
correlation = get_elevation_temp_correlation(MD_agric_df)

print(correlation)

0.20306114718387722


A Pearson correlation of 0.203 indicates a weak positive linear relationship between elevation and average temperature in this dataset. As elevation increases, temperature tends to increase slightly, but the relationship is not strong.

## __Challenge 6: Yield Correlations__

A correlation matrix measures how strongly each numerical variable relates to every other. Values close to 1 or -1 indicate a strong linear relationship. Here we ask: what actually drives `Standard_yield`?

### __Task__
Write a function `get_yield_correlations` that:
    - Considers these feature columns only: `['Elevation', 'Slope', 'Rainfall', 'Min_temperature_C', 'Max_temperature_C', 'Ave_temps', 'Soil_fertility', 'pH', 'Pollution_level', 'Plot_size', 'Annual_yield']`.
- Computes each feature's Pearson correlation with `Standard_yield`.
- Returns the result as a pandas Series sorted in **descending order of absolute value** (strongest relationship first).

> ⚠️ Do not change the function name `get_yield_correlations`.

### __Expected Output__
    ```
    Pollution_level     -0.285761
    Annual_yield         0.220812
    pH                  -0.196613
    Min_temperature_C    0.144233
    Elevation            0.129248
    Max_temperature_C   -0.111649
    Soil_fertility       0.070205
    Slope                0.056991
    Rainfall             0.039217
    Plot_size           -0.017014
    Ave_temps            0.006786
    Name: Standard_yield, dtype: float64
    
    ```

In [15]:
def get_yield_correlations(df):
    """
    Calculate Pearson correlations between selected features and Standard_yield.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing agricultural features.

    Returns
    -------
    pandas.Series
        Correlations with Standard_yield sorted by absolute value descending.
    """

    features = [
        "Elevation",
        "Slope",
        "Rainfall",
        "Min_temperature_C",
        "Max_temperature_C",
        "Ave_temps",
        "Soil_fertility",
        "pH",
        "Pollution_level",
        "Plot_size",
        "Annual_yield"
    ]

    correlations = (
        df[features + ["Standard_yield"]]
        .corr()["Standard_yield"]
        .drop("Standard_yield")
    )

    correlations = correlations.loc[
        correlations.abs().sort_values(ascending=False).index
    ]

    return correlations

In [16]:
yield_correlations = get_yield_correlations(MD_agric_df)

print(yield_correlations)

Pollution_level     -0.285761
Annual_yield         0.220812
pH                  -0.196613
Min_temperature_C    0.144233
Elevation            0.129248
Max_temperature_C   -0.111649
Soil_fertility       0.070205
Slope                0.056991
Rainfall             0.039217
Plot_size           -0.017014
Ave_temps            0.006786
Name: Standard_yield, dtype: float64
